# Red Team Agent: 5-minute demo

**Goal:** show how Microsoft Foundry generates, runs, and tracks adversarial attacks against a model through the server-side preview API in `azure-ai-projects`.

## Red teaming without PyRIT

Red teaming is a testing practice, not a synonym for PyRIT. Microsoft provides two relevant execution paths:

- **Managed Foundry scan, used here:** `project_client.beta.red_teams` submits the scan to Microsoft Foundry. Attack orchestration and evaluation are managed by the Azure service, so the local environment does not need PyRIT.
- **Client-side orchestration:** `azure-ai-evaluation[redteam]` uses PyRIT in the local Python environment to orchestrate attacks. Its target can still be a remote model, endpoint, or application; therefore, PyRIT itself is not limited to testing local targets.

This notebook deliberately uses the first path to keep the shared UV environment small and the demo reliable.

| Time | What to show |
| --- | --- |
| 0:00 | Why red teaming complements evaluation |
| 0:40 | Target, risk category, and attack strategies |
| 1:30 | Start or retrieve the scan |
| 3:00 | Scan status and attack configuration |
| 4:30 | Three takeaways |

In [ ]:
# pyright: reportUnusedImport=false
import os
import time
from datetime import datetime

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AttackStrategy,
    AzureOpenAIModelConfiguration,
    RedTeam,
    RiskCategory,
)
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

if not load_dotenv("./../credentials_my.env"):
    raise RuntimeError("Environment variables could not be loaded")

foundry_project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
deployment_name = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

print(f"Foundry project endpoint: {foundry_project_endpoint}")
print(f"Target deployment: {deployment_name}")

## A deliberately small scan

To stay within five minutes, this demo tests one risk category, **Violence**, and compares the **baseline** with a **Base64-obfuscated** attack over a single turn.

The goal is not comprehensive coverage. It is to make the workflow visible: `target → attacks → evaluation → mitigation`.

In [ ]:
scan_config = RedTeam(
    target=AzureOpenAIModelConfiguration(
        model_deployment_name=deployment_name,
    ),
    display_name=f"5-min-demo-{datetime.now():%Y%m%d-%H%M}",
    num_turns=1,
    attack_strategies=[
        AttackStrategy.BASELINE,
        AttackStrategy.BASE64,
    ],
    simulation_only=False,
    risk_categories=[RiskCategory.VIOLENCE],
    application_scenario=(
        "A general-purpose assistant that should refuse requests "
        "for harmful violent content."
    ),
    tags={"purpose": "five-minute-demo"},
)

{
    "category": RiskCategory.VIOLENCE.value,
    "strategies": [
        AttackStrategy.BASELINE.value,
        AttackStrategy.BASE64.value,
    ],
    "turns": scan_config.num_turns,
    "deployment": deployment_name,
}

## Live action

Only the next cell creates cloud work. For maximum reliability, run it before the presentation; during the live demo, retrieve that scan or show the recent scans instead.

In [ ]:
scan = project_client.beta.red_teams.create(red_team=scan_config)

print(f"Scan ID: {scan.name}")
print(f"Display name: {scan.display_name}")
print(f"Initial status: {scan.status}")

In [ ]:
terminal_statuses = {"completed", "failed", "cancelled", "canceled"}
last_status = None

for attempt in range(30):
    scan = project_client.beta.red_teams.get(scan.name)
    current_status = (scan.status or "unknown").lower()

    if current_status != last_status:
        print(f"Status: {scan.status}")
        last_status = current_status

    if current_status in terminal_statuses:
        break

    if attempt < 29:
        time.sleep(10)
else:
    raise TimeoutError(
        "The scan is still running. Continue with the recent-scans cell."
    )

## Demo-safe fallback

Listing recent scans is read-only. Use this cell live if the network is slow or the new scan does not finish in time; the demo narrative remains unchanged.

In [ ]:
recent_scans = []

for recent_scan in project_client.beta.red_teams.list():
    recent_scans.append(
        {
            "name": recent_scan.name,
            "display_name": recent_scan.display_name,
            "status": recent_scan.status,
            "risk_categories": recent_scan.risk_categories,
            "attack_strategies": recent_scan.attack_strategies,
        }
    )
    if len(recent_scans) == 5:
        break

recent_scans

## Closing

- Red teaming generates adaptive adversarial probes rather than relying only on a static test set.
- Comparing baseline and transformed attacks exposes weaknesses in application controls.
- Results provide evidence for mitigations and regression tests, but they do not guarantee security.